In [ ]:
from gitsource import GithubRepositoryDataReader
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)


documents = [file.parse() for file in reader.read()]
texts = [doc["content"] + " " + doc["filename"] for doc in documents]



In [ ]:
from embedder import Embedder
from sqlitsearch_vectorsearch import RAGVector,VectorSearchIndex
from tqdm.auto import tqdm
from openai import OpenAI
import numpy as np
embedObj = Embedder()
query='How does approximate nearest neighbor search work?'
content=''
for doc in documents:
    if doc["filename"]=='02-vector-search/lessons/07-sqlitesearch-vector.md':
        content=doc['content']
        break
    
    
vec=embedObj.encode(query)
vec2=embedObj.encode(content)

vec.dot(vec2)


In [ ]:

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

texts = [doc["content"] + " " + doc["filename"] for doc in chunks]
X=[]

batch_vectors = model.encode(texts)
X.extend(batch_vectors)

X=np.array(X)
scores = np.argmax(X.dot(vec))

chunks[scores]


In [124]:
from minsearch import VectorSearch,Index

def min_VectorSearch(query):
    
    vindex = VectorSearch(keyword_fields=["content"])
    vindex.fit(X,chunks)

    query_vector = embedObj.encode(query)

    vector_results = vindex.search(query_vector,num_results=5)

    return  vector_results



def min_IndexSearch(query):

    iindex=Index(keyword_fields=["filename"],text_fields=["content"])
    iindex.fit(chunks)

    text_results=iindex.search(query,num_results=5)

    return text_results

query='What metric do we use to evaluate a search engine?'
print(min_VectorSearch(query))


[{'start': 0, 'content': "# Evaluation\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=eC_IcxfxoiQ&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous modules, we built search engines and RAG pipelines.\nWe tried different approaches: keyword search with minsearch, vector\nsearch, agents with function calling. But we never answered the obvious\nquestion of which one is actually better.\n\nWe could try a few queries by hand and see what looks good. That's fine\nfor a quick sanity check, but it doesn't scale, and it doesn't give us a\nnumber to compare. We need a systematic way to tell whether one approach\nbeats another.\n\nThat's what evaluation is for. And it's worth saying up front: of\neverything in this course, evaluation is the part that matters most. It's\nalso the most tedious. But it's the only way to be sure your system\nworks. And it's how you keep it working as you change prompts and swap\nmodels.\n\n## The evaluation setup\n\nFor search evaluation, we 

In [126]:
query='How do I store vectors in PostgreSQL?'
vector_results=min_VectorSearch(query)
text_results=min_IndexSearch(query)

In [128]:
query='How do I give the model access to tools?'
vector_results=min_VectorSearch(query)
text_results=min_IndexSearch(query)
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

results = rrf([vector_results, text_results])

results

[{'start': 0,
  'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can